# TechRAG Architect — RAG Pipeline Notebook

This notebook documents and demonstrates the complete Retrieval-Augmented Generation (RAG) pipeline used by the project.

It covers project setup, PDF loading, cleaning, chunking, embeddings, ChromaDB indexing, retrieval, reranking, prompting, grounded generation, citation validation, evaluation results, failure analysis, and persisted vector-store configuration.

## 1. Project Setup

The notebook reuses the same modular Python code used by the production FastAPI backend rather than duplicating a second notebook-only implementation.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Could not locate project root containing src/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Vector store directory:", VECTOR_STORE_DIR)

Project root: c:\Users\abdel\OneDrive\Desktop\RAG-Powered Document
Raw data directory: c:\Users\abdel\OneDrive\Desktop\RAG-Powered Document\data\raw
Vector store directory: c:\Users\abdel\OneDrive\Desktop\RAG-Powered Document\data\vector_store


## 2. Import Project Modules

The ingestion pipeline uses `pdf_loader.py`, `cleaner.py`, `chunker.py`, and `indexer.py`. The RAG stage uses the project's embedding, retrieval, reranking, prompting, generation, citation-validation, and grounded-generation modules.

In [2]:
from src.ingestion.pdf_loader import discover_pdf_files, load_pdf, load_all_pdfs
from src.ingestion.cleaner import clean_text, clean_pages
from src.ingestion.chunker import DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP, split_text, chunk_pages
from src.ingestion.indexer import ChromaIndexer

from src.rag.embeddings import EmbeddingService, DEFAULT_EMBEDDING_MODEL
from src.rag.retrieval import RetrievalService
from src.rag.reranker import RerankerService, DEFAULT_RERANKER_MODEL
from src.rag.prompting import build_rag_prompt
from src.rag.generation import GenerationService, DEFAULT_MODEL
from src.rag.citation_validator import validate_citations
from src.rag.grounded_generation import GroundedGenerationService

print("Imports completed successfully.")

C:\Users\abdel\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Imports completed successfully.


## 3. Inspect the Document Corpus

The PDF loader recursively discovers PDFs under `data/raw/`. The first folder below `data/raw/` becomes the category.

In [3]:
pdf_files = discover_pdf_files(RAW_DATA_DIR)
print(f"Number of PDF files: {len(pdf_files)}")
for path in pdf_files[:10]:
    print(path.relative_to(RAW_DATA_DIR))
if len(pdf_files) > 10:
    print(f"... and {len(pdf_files)-10} more")

Number of PDF files: 38
ai_ml\Foundations of Machine Learning.pdf
ai_ml\Machine-Learning-With-PyTorch-and-Scikit-Learn.pdf
algorithms_data_structures\Cormen Introduction to Algorithms.pdf
algorithms_data_structures\Grokking Algorithms 2nd Edition.pdf
cloud_security\Securing_the_Cloud.pdf
computer_vision\OpenCv Computer Vision Projects with Python.pdf
computer_vision\Pattern Recognition and Machine Learning.pdf
computer_vision\Practical Python and OpenCV, 3rd Edition.pdf
computer_vision\ProgrammingComputerVision_CCdraft.pdf
cybersecurity\blackhatpython.pdf
... and 28 more


In [4]:
category_counts = {}
for path in pdf_files:
    relative = path.relative_to(RAW_DATA_DIR)
    category = relative.parts[0] if len(relative.parts) >= 2 else "uncategorized"
    category_counts[category] = category_counts.get(category, 0) + 1

category_df = pd.DataFrame(sorted(category_counts.items()), columns=["category", "pdf_count"])
category_df

,category,pdf_count
0,ai_ml,2
1,algorithms_data_structures,2
2,cloud_security,1
3,computer_vision,4
4,cybersecurity,7
5,data_science,3
6,deep_learning,6
7,embedded_systems,1
8,llm,1
9,malware_analysis,1


## 4. Load and Inspect a Sample PDF

PDFs are loaded page-by-page with `pypdf`. Successful pages retain category, document, path, and page metadata. Empty/scanned pages and extraction failures are recorded as issues.

In [5]:
if not pdf_files:
    raise RuntimeError("No PDFs found in data/raw/")

sample_pdf = pdf_files[0]
sample_pages, sample_issues = load_pdf(sample_pdf, RAW_DATA_DIR)

print("Sample PDF:", sample_pdf.name)
print("Extracted pages:", len(sample_pages))
print("Issues:", len(sample_issues))

if sample_pages:
    print("\nMetadata:", sample_pages[0]["metadata"])
    print("\nFirst 800 characters:")
    print(sample_pages[0]["text"][:800])

Sample PDF: Foundations of Machine Learning.pdf
Extracted pages: 495
Issues: 10

Metadata: {'category': 'ai_ml', 'document': 'Foundations of Machine Learning.pdf', 'path': 'c:\\Users\\abdel\\OneDrive\\Desktop\\RAG-Powered Document\\data\\raw\\ai_ml\\Foundations of Machine Learning.pdf', 'page': 2}

First 800 characters:
Foundations of Machine Learning
second edition


## 5. Full PDF Loading

The production loader can ingest the entire corpus. The cell is left optional because the project already has a persisted vector store and reprocessing every PDF on every notebook run is unnecessary.

In [6]:
# OPTIONAL: full corpus loading
# all_pages, all_issues = load_all_pdfs(RAW_DATA_DIR)
# print("Total extracted pages:", len(all_pages))
# print("Total loading issues:", len(all_issues))

## 6. Text Cleaning

The cleaner fixes simple hyphenated line breaks and normalizes whitespace while preserving paragraph boundaries. The approach is intentionally conservative so technical terms, code, formulas, and identifiers are not aggressively altered.

In [7]:
raw_example = """Machine-
learning is widely used.

This    line has     irregular spacing."""
print("Before:")
print(raw_example)
print("\nAfter:")
print(clean_text(raw_example))

Before:
Machine-
learning is widely used.

This    line has     irregular spacing.

After:
Machinelearning is widely used.

This line has irregular spacing.


In [8]:
if sample_pages:
    sample_cleaned_pages = clean_pages(sample_pages)
    print("Pages before cleaning:", len(sample_pages))
    print("Pages after cleaning:", len(sample_cleaned_pages))
    print("\nCleaned sample:")
    print(sample_cleaned_pages[0]["text"][:800])

Pages before cleaning: 495
Pages after cleaning: 495

Cleaned sample:
Foundations of Machine Learning
second edition


## 7. Chunking Strategy

Production settings:

- Chunk size: **1200 characters**
- Chunk overlap: **200 characters**
- Deterministic chunk IDs based on document, page, and page-local chunk index

A 1200-character chunk preserves enough technical context for retrieval while keeping passages focused. A 200-character overlap reduces boundary information loss. Chunking is page-aware so source page numbers remain available for citations.

In [9]:
print("Configured chunk size:", DEFAULT_CHUNK_SIZE)
print("Configured overlap:", DEFAULT_CHUNK_OVERLAP)

demo_text = ("Retrieval-Augmented Generation combines retrieval with generation. " * 40)
demo_chunks = split_text(demo_text)
print("Demo chunks:", len(demo_chunks))
for i, chunk in enumerate(demo_chunks[:3]):
    print(f"Chunk {i}: length={len(chunk)}")

Configured chunk size: 1200
Configured overlap: 200
Demo chunks: 3
Chunk 0: length=1200
Chunk 1: length=1200
Chunk 2: length=679


In [10]:
if sample_pages:
    sample_cleaned_pages = clean_pages(sample_pages)
    sample_chunks = chunk_pages(sample_cleaned_pages)
    print("Sample chunks:", len(sample_chunks))
    if sample_chunks:
        print("Chunk ID:", sample_chunks[0]["id"])
        print("Metadata:", sample_chunks[0]["metadata"])
        print("\nText:")
        print(sample_chunks[0]["text"][:800])

Sample chunks: 1140
Chunk ID: chunk_56cd4f58870d61a2
Metadata: {'category': 'ai_ml', 'document': 'Foundations of Machine Learning.pdf', 'path': 'c:\\Users\\abdel\\OneDrive\\Desktop\\RAG-Powered Document\\data\\raw\\ai_ml\\Foundations of Machine Learning.pdf', 'page': 2, 'chunk_id': 'chunk_56cd4f58870d61a2', 'page_chunk_index': 0, 'chunk_size': 46}

Text:
Foundations of Machine Learning
second edition


## 8. Embeddings

The project uses `sentence-transformers/all-MiniLM-L6-v2`. The embedding service automatically selects CUDA when available and produces normalized vectors suitable for cosine similarity.

In [11]:
embedding_service = EmbeddingService()
print("Embedding model:", DEFAULT_EMBEDDING_MODEL)
embedding = embedding_service.encode_one("What is SQL injection?")
print("Embedding shape:", embedding.shape)
print("First 10 values:", embedding[:10])

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding device: cuda
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (384,)
First 10 values: [ 0.01198135 -0.02676821 -0.08289203  0.07352556 -0.11885854 -0.04008487
  0.1149295  -0.00932092 -0.04518353 -0.01114433]


## 9. Persistent ChromaDB Vector Store

The indexer uses persistent ChromaDB with cosine distance. It sanitizes text and metadata, embeds in batches, performs upserts, and falls back to individual chunks if a batch fails.

In [12]:
indexer = ChromaIndexer(VECTOR_STORE_DIR)
print("Vector-store path:", VECTOR_STORE_DIR)
print("Indexed chunks:", indexer.count())
print("Collection exists:", indexer.collection_exists())

Vector-store path: c:\Users\abdel\OneDrive\Desktop\RAG-Powered Document\data\vector_store
Indexed chunks: 42124
Collection exists: True


In [ ]:
# OPTIONAL: rebuild/index from scratch
# all_pages, all_issues = load_all_pdfs(RAW_DATA_DIR)
# cleaned_pages = clean_pages(all_pages)
# chunks = chunk_pages(cleaned_pages)
# indexer.index_chunks(chunks, embedding_service)
# print("Indexed chunks:", indexer.count())

## 10. Retrieval and Reranking

Retrieval first performs semantic vector search in ChromaDB. Candidate passages are then reranked with `cross-encoder/ms-marco-MiniLM-L-6-v2` for better ordering.

In [13]:
reranker_service = RerankerService()
retrieval_service = RetrievalService(
    persist_directory=VECTOR_STORE_DIR,
    embedding_service=embedding_service,
    reranker_service=reranker_service,
)

print("Reranker:", DEFAULT_RERANKER_MODEL)
print("Available categories:", retrieval_service.get_categories())
print("Total chunks:", retrieval_service.count())

Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Available categories: ['ai_ml', 'algorithms_data_structures', 'cloud_security', 'computer_vision', 'cybersecurity', 'data_science', 'deep_learning', 'embedded_systems', 'llm', 'malware_analysis', 'nlp', 'python']
Total chunks: 42124


In [14]:
test_query = "What is SQL injection?"
results = retrieval_service.retrieve(
    query=test_query,
    top_k=5,
    candidate_k=15,
    category="cybersecurity",
    min_similarity=0.30,
    use_reranker=True,
)

for result in results:
    metadata = result.get("metadata", {})
    print("=" * 80)
    print("Rank:", result.get("rank"))
    print("Document:", metadata.get("document"))
    print("Page:", metadata.get("page"))
    print("Category:", metadata.get("category"))
    print("Similarity:", round(float(result.get("similarity", 0)), 4))
    print("Reranker score:", result.get("reranker_score"))
    print(result.get("text", "")[:500])

Rank: 1
Document: Web Hacking 101 - How to Make Money Hacking Ethically by Peter Yaworski.pdf
Page: 90
Category: cybersecurity
Similarity: 0.733
Reranker score: 7.960837364196777
11. SQL Injection
Description
A structured query language (SQL) injection, or SQLi, occurs when a vulnerability on a
database-backed site allows an attacker to query or otherwise attack the site’s database.
SQLi attacks are often highly rewarded because they can be devastating. They can enable
an attacker to manipulate or extract information or even create an administrator log in
for themselves in the database.
SQL Databases
Databases store information in records and fields contained in a collec
Rank: 2
Document: Dafydd Stuttard, Marcus Pinto - The web application hacker's handbook_ finding and exploiting security flaws-Wiley (2011).pdf
Page: 327
Category: cybersecurity
Similarity: 0.6385
Reranker score: 6.370115756988525
cted web applications. In the most 
serious cases, SQL injection can enable an anonymous 

## 11. Retrieval Test Set

This test set demonstrates retrieval across more than ten questions and multiple categories.

In [15]:
retrieval_test_cases = [
    ("What is overfitting in machine learning?", "ai_ml"),
    ("What is regularization in machine learning?", "ai_ml"),
    ("What is a binary search tree?", "algorithms_data_structures"),
    ("What is SQL injection?", "cybersecurity"),
    ("What is cross-site scripting?", "cybersecurity"),
    ("What is a pandas DataFrame?", "data_science"),
    ("What is static malware analysis?", "malware_analysis"),
    ("What is a Python decorator?", "python"),
    ("What is transfer learning?", "deep_learning"),
    ("What is object detection?", "computer_vision"),
    ("What is cloud security?", "cloud_security"),
    ("What is an embedded system?", "embedded_systems"),
    ("What is a large language model?", "llm"),
    ("What is tokenization in NLP?", "nlp"),
]

rows = []
for question, category in retrieval_test_cases:
    retrieved = retrieval_service.retrieve(
        query=question,
        top_k=3,
        candidate_k=10,
        category=category,
        min_similarity=None,
        use_reranker=True,
    )
    top = retrieved[0] if retrieved else None
    rows.append({
        "question": question,
        "category": category,
        "retrieved": bool(retrieved),
        "top_document": top.get("metadata", {}).get("document") if top else None,
        "top_page": top.get("metadata", {}).get("page") if top else None,
        "top_similarity": round(float(top.get("similarity", 0)), 4) if top else None,
    })

retrieval_demo_df = pd.DataFrame(rows)
retrieval_demo_df

,question,category,retrieved,top_document,top_page,top_similarity
0,What is overfitting in machine learning?,ai_ml,True,Machine-Learning-With-PyTorch-and-Scikit-Learn...,102,0.6908
1,What is regularization in machine learning?,ai_ml,True,Machine-Learning-With-PyTorch-and-Scikit-Learn...,103,0.6370
2,What is a binary search tree?,algorithms_data_structures,True,Cormen Introduction to Algorithms.pdf,308,0.6649
3,What is SQL injection?,cybersecurity,True,Web Hacking 101 - How to Make Money Hacking Et...,90,0.7330
4,What is cross-site scripting?,cybersecurity,True,Web Hacking 101 - How to Make Money Hacking Et...,20,0.5707
5,What is a pandas DataFrame?,data_science,True,Python-for-Data-Analysis.pdf,23,0.6340
6,What is static malware analysis?,malware_analysis,True,practicalmalwareanalysis.pdf,42,0.7852
7,What is a Python decorator?,python,True,Serious Python.pdf,136,0.7829
8,What is transfer learning?,deep_learning,True,Programming PyTorch for Deep Learning.pdf,70,0.4996
9,What is object detection?,computer_vision,True,OpenCv Computer Vision Projects with Python.pdf,277,0.6550


## 12. Prompt Construction

The production prompt is context-only. It instructs the LLM to avoid outside knowledge, avoid invented documents/pages, use only allowed citations, and admit when the provided context is insufficient.

In [16]:
if not results:
    raise RuntimeError("No retrieval results available for prompt demonstration.")

prompt = build_rag_prompt(
    query=test_query,
    retrieved_chunks=results,
)

print(prompt[:5000])

You are TechRAG Architect, a grounded technical document assistant.

Your answer MUST be based ONLY on the retrieved context supplied below.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent facts, document names, or page numbers.
3. If the retrieved context is insufficient, explicitly say:
   "I don't have enough information in the provided documents to answer this reliably."
4. Every factual paragraph must contain at least one supporting citation.
5. Use ONLY citations from the AVAILABLE VALID CITATIONS list.
6. Copy citations EXACTLY as provided.
7. Never modify a document filename.
8. Never modify or guess a page number.
9. Never cite a source that is not listed in AVAILABLE VALID CITATIONS.
10. Do not mention chunk IDs.
11. Answer clearly and concisely.


USER QUESTION:
What is SQL injection?

AVAILABLE VALID CITATIONS:
- [Document: Web Hacking 101 - How to Make Money Hacking Ethically by Peter Yaworski.pdf, Page: 90]
- [Document: Dafydd Stuttard, Marcus Pinto - Th

## 13. Local LLM Generation

The project uses `gemma3:4b` through Ollama. Generation uses a low temperature to reduce unnecessary variability.

In [17]:
generation_service = GenerationService(model_name=DEFAULT_MODEL)
print("Generation model:", generation_service.model_name)

# OPTIONAL direct generation
# answer = generation_service.generate(prompt=prompt, temperature=0.1)
# print(answer)

Generation model: gemma3:4b


## 14. Grounded Generation and Citation Validation

The grounded-generation service validates generated citations against the retrieved document/page pairs. If citations are missing or invalid, the service can retry with stricter instructions.

In [18]:
grounded_service = GroundedGenerationService(
    generation_service=generation_service,
    max_retries=2,
)

# OPTIONAL complete grounded-generation example
# grounded_result = grounded_service.generate_grounded_answer(
#     query=test_query,
#     retrieved_chunks=results,
# )
# print("Answer:")
# print(grounded_result["answer"])
# print("Attempts:", grounded_result["attempts"])
# print("Success:", grounded_result["success"])
# print("Validation:", grounded_result["validation"])

## 15. End-to-End Evaluation Results

The project includes `scripts/evaluate_rag_full.py` for end-to-end testing.

Latest result:

| Metric | Result |
|---|---:|
| Auto-routing correct | 20 / 21 |
| Grounded answers | 20 / 21 |
| Valid citations | 20 / 21 |
| Overall passed | 19 / 21 |
| Overall pass rate | **90.48%** |

The script writes results to `evaluation/rag_evaluation_results.csv` and `evaluation/rag_evaluation_summary.md`.

In [19]:
evaluation_csv = PROJECT_ROOT / "evaluation" / "rag_evaluation_results.csv"

if evaluation_csv.exists():
    evaluation_df = pd.read_csv(evaluation_csv)
    display(evaluation_df)
else:
    print("Evaluation CSV not found. Run: python -m scripts.evaluate_rag_full")

,question,expected_category,resolved_category,category_correct,needs_category_selection,grounding_success,citation_valid,citation_count,top_document,top_page,top_similarity,top_reranker_score,answer,overall_pass,error
0,What is overfitting in machine learning?,ai_ml,ai_ml,True,False,True,True,3,Machine-Learning-With-PyTorch-and-Scikit-Learn...,102,0.690849,8.292717,Overfitting is a common problem in machine lea...,True,NaN
1,What is regularization in machine learning?,ai_ml,ai_ml,True,False,True,True,5,Machine-Learning-With-PyTorch-and-Scikit-Learn...,103,0.637017,8.790302,Regularization in machine learning is a method...,True,NaN
2,What is dynamic programming?,algorithms_data_structures,ai_ml,False,False,True,True,3,Machine-Learning-With-PyTorch-and-Scikit-Learn...,706,0.785729,9.738111,Dynamic programming is about recursive problem...,False,NaN
3,What is a binary search tree?,algorithms_data_structures,algorithms_data_structures,True,False,True,True,4,Cormen Introduction to Algorithms.pdf,308,0.664902,8.215016,A binary search tree is organized as a binary ...,True,NaN
4,What is SQL injection?,cybersecurity,cybersecurity,True,False,False,False,0,Web Hacking 101 - How to Make Money Hacking Et...,90,0.733015,7.960837,I found relevant information in the retrieved ...,False,NaN
5,What is cross-site scripting?,cybersecurity,cybersecurity,True,False,True,True,4,Web Hacking 101 - How to Make Money Hacking Et...,20,0.570689,6.401739,Cross-site scripting (XSS) represents huge opp...,True,NaN
6,What is penetration testing?,cybersecurity,cybersecurity,True,False,True,True,3,Penetration Testing - 3 Manuscripts—Wireless H...,15,0.672146,8.608316,Penetration testing is simulating real attacks...,True,NaN
7,What is privilege escalation?,cybersecurity,cybersecurity,True,False,True,True,4,blackhatpython.pdf,159,0.479791,5.397005,Privilege escalation is the process of gaining...,True,NaN
8,What is a pandas DataFrame?,data_science,data_science,True,False,True,True,2,Python Data Science Handbook.pdf,115,0.557412,6.726852,A pandas DataFrame is a multidimensional array...,True,NaN
9,What is exploratory data analysis?,data_science,data_science,True,False,True,True,3,Python-for-Data-Analysis.pdf,19,0.527545,-4.534443,Exploratory data analysis involves investigati...,True,NaN


## 16. Failure Analysis

### Dynamic programming
Expected category: `algorithms_data_structures`  
Resolved category: `ai_ml`

This is an Auto-routing error caused by semantic overlap between algorithmic concepts and machine-learning material. The generation remained grounded, but the selected category was incorrect.

### SQL injection
The router correctly selected `cybersecurity`, but one evaluation run failed grounding/citation validation. This reflects stochastic LLM generation rather than a retrieval-routing error.

### Mitigations

- Hybrid Auto-routing combines document evidence with semantic category descriptions.
- Manual category selection is available when needed.
- CrossEncoder reranking improves final context ordering.
- Citation validation rejects unsupported or malformed citations.
- Grounded generation retries citation failures.
- Insufficient context produces a safe fallback rather than fabricated information.

## 17. Persisted Vector Store

The ChromaDB store is persisted under `data/vector_store/`, so embeddings do not need to be recomputed when the backend starts. The vector store can be excluded from Git and reproduced from the local corpus.

In [20]:
print("Vector store exists:", VECTOR_STORE_DIR.exists())
print("Indexed chunk count:", retrieval_service.count())

Vector store exists: True
Indexed chunk count: 42124


## 18. Pipeline Summary

```text
PDF Documents
    ↓
PDF Discovery & Page Extraction
    ↓
Text Cleaning
    ↓
1200-character Chunks + 200-character Overlap
    ↓
MiniLM Embeddings
    ↓
Persistent ChromaDB
    ↓
Semantic Retrieval
    ↓
CrossEncoder Reranking
    ↓
Context-Only Prompt
    ↓
Gemma 3 4B via Ollama
    ↓
Citation Validation / Retry
    ↓
Grounded Answer + Sources
```